In [11]:
import os
import sys

# set working directory to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
sys.path.insert(0, project_root)


In [4]:
import logging
import os

from dotenv import load_dotenv
from opensearchpy import OpenSearch

from src.ingestion.indexer import create_index
from src.ingestion.loader import load_all_pdfs

In [5]:
load_dotenv()

True

In [6]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

In [7]:
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "nomic-embed-text")
DOCS_DIR = "docs/raw"
CHUNK_SIZE = 800
OVERLAP = 100
SKIP_EXISTING = False

In [8]:
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_compress=True
)

create_index(client)
print("Index created successfully")

2026-07-29 23:20:34,439 - opensearch - INFO - PUT http://localhost:9200/angioedema [status:200 request:0.247s]
2026-07-29 23:20:34,439 - src.ingestion.indexer - INFO - Index 'angioedema' created successfully


Index created successfully


In [9]:
documents = load_all_pdfs(DOCS_DIR)
print(f"Loaded {len(documents)} documents")

2026-07-29 23:20:34,451 - src.ingestion.loader - INFO - Found 10 PDFs in docs\raw
2026-07-29 23:20:34,451 - src.ingestion.loader - INFO - Loading PDF: ANVISA - Cinryze.pdf
2026-07-29 23:20:34,542 - src.ingestion.loader - INFO - Extracted: 1/1 pages with text from ANVISA - Cinryze.pdf
2026-07-29 23:20:34,544 - src.ingestion.loader - INFO - Loading PDF: ANVISA - Icatibanto.pdf
2026-07-29 23:20:34,965 - src.ingestion.loader - INFO - Extracted: 151/151 pages with text from ANVISA - Icatibanto.pdf
2026-07-29 23:20:34,967 - src.ingestion.loader - INFO - Loading PDF: ANVISA - Lanadelumabe.pdf
2026-07-29 23:20:35,359 - src.ingestion.loader - INFO - Extracted: 91/93 pages with text from ANVISA - Lanadelumabe.pdf
2026-07-29 23:20:35,361 - src.ingestion.loader - INFO - Loading PDF: ANVISA -ANDEMBRY®.pdf
2026-07-29 23:20:35,378 - src.ingestion.loader - INFO - Extracted: 1/1 pages with text from ANVISA -ANDEMBRY®.pdf
2026-07-29 23:20:35,378 - src.ingestion.loader - INFO - Loading PDF: ASBAI - Angio

Loaded 10 documents


In [10]:
from src.ingestion.chunker import RecursiveChunker
from src.ingestion.indexer import index_chunks

chunker = RecursiveChunker(chunk_size=CHUNK_SIZE, overlap=OVERLAP)
total_indexed = 0
total_failed = 0
total_skipped = 0

for doc in documents:
    print(f"\nProcessing: {doc.metadata['source']}")
    chunks = chunker.chunk(doc)

    summary = index_chunks(
        chunks=chunks,
        client=client,
        ollama_url=OLLAMA_URL,
        embedding_model=EMBEDDING_MODEL,
        batch_size=10,
        skip_existing=SKIP_EXISTING
    )

    total_indexed += summary["indexed"]
    total_failed += summary["failed"]
    total_skipped += summary["skipped"]

print(f"\n{'='*50}")
print("Pipeline complete")
print(f"Total indexed:  {total_indexed}")
print(f"Total skipped:  {total_skipped}")
print(f"Total failed:   {total_failed}")

2026-07-29 23:20:36,084 - src.ingestion.chunker - INFO - [Recursive] Chunking ANVISA - Cinryze.pdf (size=800, overlap=100)
2026-07-29 23:20:36,177 - src.ingestion.chunker - INFO - [Recursive] Generated 2 chunks
2026-07-29 23:20:36,180 - opensearch - INFO - HEAD http://localhost:9200/angioedema [status:200 request:0.004s]
2026-07-29 23:20:36,253 - opensearch - INFO - POST http://localhost:9200/angioedema/_delete_by_query [status:200 request:0.072s]



Processing: ANVISA - Cinryze.pdf


2026-07-29 23:20:36,254 - src.ingestion.indexer - INFO - Deleted 0 existing chunks for source: ANVISA - Cinryze.pdf
2026-07-29 23:20:36,254 - src.ingestion.indexer - INFO - Starting indexing of 2 chunks in batches of 10 (retry mode: False)
2026-07-29 23:20:42,593 - opensearch - INFO - POST http://localhost:9200/_bulk [status:200 request:0.641s]
2026-07-29 23:20:42,593 - src.ingestion.indexer - INFO - Progress: 2/2 chunks processed
2026-07-29 23:20:42,593 - src.ingestion.indexer - INFO - Indexing complete: {'total': 2, 'indexed': 2, 'failed': 0, 'skipped': 0}
2026-07-29 23:20:42,598 - src.ingestion.chunker - INFO - [Recursive] Chunking ANVISA - Icatibanto.pdf (size=800, overlap=100)



Processing: ANVISA - Icatibanto.pdf


2026-07-29 23:20:42,949 - src.ingestion.chunker - INFO - [Recursive] Generated 215 chunks
2026-07-29 23:20:42,949 - opensearch - INFO - HEAD http://localhost:9200/angioedema [status:200 request:0.000s]
2026-07-29 23:20:43,008 - opensearch - INFO - POST http://localhost:9200/angioedema/_delete_by_query [status:200 request:0.059s]
2026-07-29 23:20:43,016 - src.ingestion.indexer - INFO - Deleted 0 existing chunks for source: ANVISA - Icatibanto.pdf
2026-07-29 23:20:43,016 - src.ingestion.indexer - INFO - Starting indexing of 215 chunks in batches of 10 (retry mode: False)
2026-07-29 23:20:50,808 - opensearch - INFO - POST http://localhost:9200/_bulk [status:200 request:0.058s]
2026-07-29 23:20:50,808 - src.ingestion.indexer - INFO - Progress: 10/215 chunks processed
2026-07-29 23:20:57,149 - opensearch - INFO - POST http://localhost:9200/_bulk [status:200 request:0.183s]
2026-07-29 23:20:57,149 - src.ingestion.indexer - INFO - Progress: 20/215 chunks processed
2026-07-29 23:21:03,783 - op

...


Processing: EBSERH - Protocolo Assistencial.pdf


2026-07-29 23:27:06,518 - opensearch - INFO - POST http://localhost:9200/_bulk [status:200 request:0.031s]
2026-07-29 23:27:06,518 - src.ingestion.indexer - INFO - Progress: 6/6 chunks processed
2026-07-29 23:27:06,525 - src.ingestion.indexer - INFO - Indexing complete: {'total': 6, 'indexed': 6, 'failed': 0, 'skipped': 0}
2026-07-29 23:27:06,525 - src.ingestion.chunker - INFO - [Recursive] Chunking HAEA - Treatment Guidelines_en.pdf (size=800, overlap=100)
2026-07-29 23:27:06,654 - src.ingestion.chunker - INFO - [Recursive] Generated 58 chunks
2026-07-29 23:27:06,669 - opensearch - INFO - HEAD http://localhost:9200/angioedema [status:200 request:0.000s]
2026-07-29 23:27:06,721 - opensearch - INFO - POST http://localhost:9200/angioedema/_delete_by_query [status:200 request:0.052s]



Processing: HAEA - Treatment Guidelines_en.pdf


2026-07-29 23:27:06,721 - src.ingestion.indexer - INFO - Deleted 0 existing chunks for source: HAEA - Treatment Guidelines_en.pdf
2026-07-29 23:27:06,721 - src.ingestion.indexer - INFO - Starting indexing of 58 chunks in batches of 10 (retry mode: False)
2026-07-29 23:27:13,857 - opensearch - INFO - POST http://localhost:9200/_bulk [status:200 request:0.046s]
2026-07-29 23:27:13,857 - src.ingestion.indexer - INFO - Progress: 10/58 chunks processed
2026-07-29 23:27:21,103 - opensearch - INFO - POST http://localhost:9200/_bulk [status:200 request:0.089s]
2026-07-29 23:27:21,103 - src.ingestion.indexer - INFO - Progress: 20/58 chunks processed
2026-07-29 23:27:27,202 - opensearch - INFO - POST http://localhost:9200/_bulk [status:200 request:0.048s]
2026-07-29 23:27:27,202 - src.ingestion.indexer - INFO - Progress: 30/58 chunks processed
2026-07-29 23:27:34,709 - opensearch - INFO - POST http://localhost:9200/_bulk [status:200 request:0.051s]
2026-07-29 23:27:34,709 - src.ingestion.indexer


Pipeline complete
Total indexed:  561
Total skipped:  0
Total failed:   0
